In [1]:
import pandas as pd

In [2]:
import numpy as np

In [3]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [4]:
df = pd.read_csv('covid_toy.csv')

In [5]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [9]:
df['city'].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [10]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [11]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],
                                                test_size=0.2)

X_train

### simple way to do without using column transfer


In [14]:
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

# also the test data
X_test_fever = si.fit_transform(X_test[['fever']])
                                 
X_train_fever.shape


(80, 1)

SimpleImputer is used to handle missing values (NaN) in data.

By default, it replaces missing values with the mean of the column.

fit() learns the statistic only from training data.

transform() applies the same statistic to new data.

Never use fit_transform() on test data → it causes data leakage.

In [ ]:
# Ordinalencoding -> cough
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

# also the test data
X_test_cough = oe.fit_transform(X_test[['cough']])

X_train_cough.shape

OrdinalEncoder is used for ordinal categorical features (categories with order).

Here, cough has an order:
Mild < Strong

categories=[['Mild','Strong']] defines the exact order:

Mild → 0

Strong → 1

fit() learns the mapping from training data only.

❌ Using fit_transform() on test data causes data leakage.

✅ Correct usage:

X_train_cough = oe.fit_transform(X_train[['cough']])
X_test_cough  = oe.transform(X_test[['cough']])


Output is a 2D NumPy array.

.shape returns (number_of_rows, 1).

✅ Use OrdinalEncoder only when category order matters.

# OneHotEncoding -> gender,city

ohe = OneHotEncoder(drop='first', sparse_output=False)

X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])
X_test_gender_city = ohe.transform(X_test[['gender','city']])

X_train_gender_city.shape


OneHotEncoder is used for nominal categorical features (no order).

Here: gender, city.

drop= 'first'removes one category per feature to avoid dummy variable trap.

fit() learns categories only from training data.

transform() applies the same encoding to test data → no data leakage.

sparse_output=False returns a NumPy array instead of sparse matrix.

Output has multiple columns (one per category − dropped ones).


In [21]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

# also the test data
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age.shape


(80, 1)

This code extracts the numerical feature(s) (age) from the dataset.

drop(columns=[...]) removes categorical and other processed features.

.values converts the DataFrame into a NumPy array (required by sklearn).

Done separately for train and test to keep data aligned.

Output is a 2D array.


In [22]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
# also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

X_train_transformed.shape


(80, 7)

np.concatenate(..., axis=1) horizontally combines all processed features.

Combines:

age (numeric)

fever (imputed)

gender, city (one-hot encoded)

cough (ordinal encoded)

axis=1 means column-wise merge.

All arrays must have the same number of rows.

Train and test are concatenated separately to maintain consistency.

Output is the final feature matrix used for model training.

### column Transfer way 

In [24]:
from sklearn.compose import ColumnTransformer


In [25]:
transformer = ColumnTransformer(
    transformers=[
        ('fever_impute', SimpleImputer(strategy='mean'), ['fever']),
        ('cough_encode', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']),
        ('gender_city_ohe', OneHotEncoder(drop='first', sparse_output=False), ['gender', 'city'])
    ],
    remainder='passthrough'
)